In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# import ollama
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
import requests
from tqdm import tqdm
import time

In [2]:
# df = pd.read_csv('./data/manually_labelled.csv')
df = pd.read_csv('./data/tenpct_comments_manual_labelled.csv')
# print(df.head(2))

print('Length of DF: ', len(df))
print(df.isna().sum())

Length of DF:  336
category                   0
post_id                    0
post_url                   0
post_date                  0
post_title                 0
comment_id                 0
comment_url                0
comment_date               0
comment_votes              0
comment_body               0
comment_has_multimedia     0
comment_has_links          0
comment_stance            56
comment_argument          78
dtype: int64


In [3]:
print(df.notna().sum())

category                  336
post_id                   336
post_url                  336
post_date                 336
post_title                336
comment_id                336
comment_url               336
comment_date              336
comment_votes             336
comment_body              336
comment_has_multimedia    336
comment_has_links         336
comment_stance            280
comment_argument          258
dtype: int64


In [4]:
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'\[.*?\]\(.*?\)', '', text)  
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)  
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return ''

def filter_bot(text):
    return isinstance(text, str) and 'i am a bot' in text.lower() #iamabot' in re.sub(r'\s+', '', text)

In [5]:
count = 0
for idx, text in df['comment_body'].items():
    cleaned_text = clean_text(text)
    if filter_bot(cleaned_text):
        count += 1
        # print(cleaned_text)
        df.drop(idx, inplace=True)

print(count)

24


In [6]:
len(df[df['comment_body'].isna() & df['comment_stance'].isna()])

0

In [7]:
df = df[~df['comment_stance'].isna()]

In [8]:
df['comment_stance'] = df['comment_stance'].replace({'neutral ': 'neutral', 'approve ': 'approve'})

In [9]:
df['comment_body'] = df['comment_body'].apply(clean_text)

In [10]:
df['new_stance'] = df['comment_stance'].replace({'approve': 'agree', 'disapprove': 'disagree'})

In [11]:
print(df['comment_stance'].value_counts())
print(df['new_stance'].value_counts())

comment_stance
disapprove    162
approve        64
neutral        54
Name: count, dtype: int64
new_stance
disagree    162
agree        64
neutral      54
Name: count, dtype: int64


In [12]:
def strip_content(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text.lower())

    try:
        stance_index = text.index('stance')
    except ValueError:
        stance_index = 0

    # try:
    #     arg_index = text.index('rationale')
    # except ValueError:
    #     arg_index = 0


    # if arg_index>stance_index:
    #     stance_text = text[stance_index:arg_index]
    # else:
    #     stance_text = text[stance_index:]
    stance_text = text[stance_index:]

    # stance = 'disapprove' if 'disagree' in stance_text else 'approve'
    # stance = 'disagree' if 'disagree' in stance_text else 'agree'
    # stance = 'disagree' if 'no' in stance_text else 'agree'
    if 'neutral' in stance_text:
        stance = 'neutral'
    elif 'disapprove' in stance_text:
        stance = 'disapprove'
    else:
        stance = 'approve'


    # argument = text[arg_index:].split(' ')
    
    # try:
    #     argument.remove('rationale')
    # except:
    #     pass

    # argument = ' '.join(argument)

    # stat_index = text.index('strategy')
    
    # stat = text[stat_index:].split(' ')
    # stat.remove('strategy')
    # stat = ' '.join(stat)

    return stance#, argument#, stat

def strip_recheck(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text.lower())
    if 'yes' in text:
        return True 
    else:
        return False

In [13]:
# LLM prompt template
# def response_prompt(title, comment):
#     return f"""You are a highly skilled political discourse analyst. Your task is to determine the relationship between a given political POST and a user COMMENT.

#     POST: "{title}"
#     COMMENT: "{comment}"

#     Evaluate the COMMENT and categorize its stance towards the POST as one of the following:

#     * **AGREE**: The COMMENT expresses support for, alignment with, or reinforcement of the views or arguments presented in the POST.
#     * **DISAGREE**: The COMMENT expresses opposition to, counter-arguments against, or rejection of the views or arguments presented in the POST.
#     * **NEUTRAL**: The COMMENT neither clearly agrees nor clearly disagrees with the POST, or the POST itself does not contain a discernible political argument to which a stance can be taken.

#     Your response must be one of these exact words: AGREE, DISAGREE, NEUTRAL.
#     """

def response_prompt(title, comment):
    return f"""
        You are a stance-classification assistant.
        Given a **post title** (claim) and a **comment**, determine if the comment:
        - **Approves** (positive stance toward the claim)
        - **Disapproves** (negative stance)
        - **Neutral** (neither supports nor opposes)

        Below, show your reasoning step-by-step, then conclude with one of: APPROVE / DISAPPROVE / NEUTRAL.

        ### Example:
        **Post:** "Electric cars are better for the environment."
        **Comment:** "I drive a diesel truck and haven't seen any benefit from electric cars."
        **Reasoning:**
        1. The comment expresses a negative opinion toward the claim (diesel preference).
        2. It indicates disapproval of the environmental benefit claim.
        **Conclusion:** DISAPPROVE

        ---

        ### Now classify:
        **Post:** "{title}"
        **Comment:** "{comment}"
    """

def recheck_prompt(title, comment, response):
    res_dict = {
        'approve': 'agree',
        'disapprove': 'disagree'
    }
    response = res_dict[response]
    return f"""
    Post Title: "{title}
    Comment: "{comment}"
    AI Response: "{response}"

    The AI response says the comment {response} with the post title, is this correct?

    Respond with only yes or no
    """



def query_ollama(prompt, mdl_idx):
    models_dict = { 0: 'gemma3:12b', 1: 'gemma3:4b', 2: 'deepseek-r1:8b', 3: 'tinyllama:1.1b'}
    url = "http://localhost:11434/api/generate"
    data = {
        "model": models_dict[mdl_idx],
        "prompt": prompt,
        "stream": False
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        print("Error:", e)
        return None

In [14]:
def apply_on_df(df):
    selected_mdl_idx = 0
    for idx, row in tqdm(df.iterrows(), desc='Processing: ', total=len(df), dynamic_ncols=True):
        prompt = response_prompt(row['post_title'], row['comment_body'])
        llm_response = query_ollama(prompt, selected_mdl_idx) 

        if llm_response:
            stance = strip_content(llm_response)
            # check_prompt = recheck_prompt(row['post_title'], row['comment_body'], stance)
            # llm_check_response = query_ollama(check_prompt, selected_mdl_idx)
            # llm_check_flag = strip_recheck(llm_check_response)

            # # print('LLM Response: \n', llm_response)
            # # print('LLM Check Response: \n', llm_check_response)
            # # print('Check Flag: ', llm_check_flag)

            # if not llm_check_flag:
            #     stance2 = 'disapprove' if stance=='approve' else 'approve'

            df.at[idx, 'llm_stance'] = stance.lower() 
            # df.at[idx, 'llm_stance2'] = stance2.lower()
            # df.at[idx, 'llm_argument'] = argument.lower() 
   
        time.sleep(1)

    return df

In [15]:
result_df = apply_on_df(df)
result_df.to_csv('data/llm_labelled_1806v1.csv')

Processing:   0%|          | 0/280 [00:00<?, ?it/s]

Processing:   3%|▎         | 9/280 [01:59<59:47, 13.24s/it]  


KeyboardInterrupt: 

In [17]:
ytrue, ypred = result_df['comment_stance'], result_df['llm_stance'].map(str.lower)
acc_score = accuracy_score(ytrue, ypred)
clf_report = classification_report(ytrue, ypred)
cm = confusion_matrix(ytrue, ypred)

print('Accuracy Score (Gemma3:12b)-0606:', acc_score)
print('classification Report:')
print(clf_report)

Accuracy Score (Gemma3:12b)-0606: 0.2357142857142857
classification Report:
              precision    recall  f1-score   support

     approve       0.25      0.62      0.36        64
  disapprove       0.67      0.05      0.09       162
     neutral       0.17      0.33      0.22        54

    accuracy                           0.24       280
   macro avg       0.36      0.34      0.22       280
weighted avg       0.47      0.24      0.18       280

